# Lab 0 — Jupyter Environment Setup

Prepare Python dependencies, verify AWS Bedrock access, and inspect synthetic incident fixtures.

**Run cells from top to bottom.** Execute `00_Setup.ipynb` first. Each notebook is intended to be opened from the `notebooks/` folder.

## 1. Open this folder as the notebook workspace

Start JupyterLab from the project root (`strands_multiagent_labs`) and open `notebooks/00_Setup.ipynb`. Run the following cell in your selected Python kernel. `!` and `%pip` are Jupyter commands. This code is used to configure the Python environment before running your Strands Agents hands-on labs in Jupyter Notebook. Its main purpose is to ensure that Jupyter runs from the correct project directory and can import Python files such as common.py

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "common.py").exists(), f"Open Jupyter from the extracted lab project; unexpected directory: {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Python:", sys.version.split()[0])

Project root: C:\Users\LeCaoThiHoang\strands_multiagent_labs
Python: 3.14.5


## 2. Install dependencies into the active notebook kernel

Run this once. **Restart your kernel afterward**, then rerun the previous cell. Python 3.10+ is required.

In [2]:
%pip install -r requirements.txt ipykernel jupyterlab

Note: you may need to restart the kernel to use updated packages.


## 3. Check AWS authentication and model access

This lab uses the SDK default Bedrock model configuration. Confirm the AWS profile, region, and that the model is enabled for inference. Do not paste AWS access keys into the notebook.

In [3]:
import boto3
session = boto3.Session()
print("AWS region:", session.region_name)
print("AWS profile:", session.profile_name or "default/environment")
try:
    ident = session.client("sts").get_caller_identity()
    print("Authenticated account:", ident["Account"])
except Exception as exc:
    print("Credential check failed:", type(exc).__name__, str(exc)[:300])
    print("Configure an AWS profile outside Jupyter and restart the kernel.")

AWS region: us-east-1
AWS profile: default
Authenticated account: 452506543602


## 4. Inspect the incident (no model call)

This code snippet is used to read and display simulated Incident information, while also checking the required components before running Strands Agents.

Important: This code snippet does not call an AI model, execute an Agent, or access AWS CloudWatch.

In [4]:
# This section imports four components from the common.py file
from common import INCIDENT, get_incident_metrics, get_incident_logs, STATE
import json
print(json.dumps(INCIDENT, indent=2))
print("State folder:", STATE)
print("Metrics tool:", get_incident_metrics.__name__ if hasattr(get_incident_metrics, "__name__") else type(get_incident_metrics).__name__)

{
  "incident_id": "INC-1042",
  "service": "checkout-api",
  "environment": "training-sandbox",
  "metric_window": "2026-11-10T08:00:00Z/2026-11-10T08:15:00Z",
  "p95_latency_ms": {
    "baseline": 180,
    "observed": 2450
  },
  "http_5xx_pct": 8.1,
  "database_connections": {
    "max": 100,
    "observed": 99
  },
  "cpu_pct": 48,
  "log_samples": [
    "08:03:10 ERROR checkout-api database connection acquisition timed out",
    "08:04:11 WARN checkout-api connection pool near capacity",
    "08:06:05 ERROR checkout-api request failed: DB pool exhausted"
  ],
  "recent_change": "checkout-api release 2.7 at 07:55 UTC"
}
State folder: C:\Users\LeCaoThiHoang\strands_multiagent_labs\state
Metrics tool: get_incident_metrics


**Checkpoint:** The project root and incident fixture display correctly. `boto3` identifies your AWS account. **Do not run subsequent model cells until Bedrock access is configured.**